# Cell 1: Setup & Imports

In [ ]:
# Cell 1: Setup & Imports
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, mean_absolute_error, cohen_kappa_score
from scipy import stats

# ============ PATHS ============
BASE = Path(".").resolve()  # Notebook location or BASE = Path(__file__).parent
RAG_BASE = BASE / "Retrieval-Augmented-Classification"
RESULTS_BASE = RAG_BASE / "results" / "gemini"

FOLDS = [1, 2, 3, 4, 5]
K = 5  # Only K=5
VARIANTS = ["variant_A", "variant_B"]

print(f"📍 Base path: {RAG_BASE}")
print(f"📍 Results path: {RESULTS_BASE}")

# Cell 2: Load Predictions

In [ ]:
# Cell 2: Load Predictions
def load_predictions(variant: str, k: int = 5) -> pd.DataFrame:
    """
    Load predictions for a given variant across all folds at k=5.
    
    Returns DataFrame with columns:
    - fold, variant, k, test_row_id, question, answer, final_decision, model_level
    """
    dfs = []
    for fold_id in FOLDS:
        csv_path = RESULTS_BASE / f"{variant}" / f"fold_{fold_id}" / f"gemini_fold{fold_id}_k{k}.csv"
        
        if not csv_path.exists():
            print(f"⚠️ Missing: {csv_path}")
            continue
        
        df = pd.read_csv(csv_path)
        df = df[['fold', 'variant', 'k', 'test_row_id', 'question', 'answer', 'final_decision', 'model_level']]
        df = df.fillna("")
        dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# Load both variants
var_A = load_predictions("variant_A", k=K)
var_B = load_predictions("variant_B", k=K)

print(f"✅ Loaded Variant A: {len(var_A)} predictions")
print(f"✅ Loaded Variant B: {len(var_B)} predictions")

all_preds = pd.concat([var_A, var_B], ignore_index=True)
display(all_preds.head(10))

# Cell 3: Compute Metrics Function

In [ ]:
# Cell 3: Compute Metrics Function
def compute_metrics(df: pd.DataFrame) -> dict:
    """
    Compute F1-Macro, F1-Weighted, QWK (Cohen's Kappa with quadratic weights), and MAE.
    
    Returns dict with keys: f1_macro, f1_weighted, qwk, mae, support
    """
    if df.empty or "final_decision" not in df or "model_level" not in df:
        return {"f1_macro": np.nan, "f1_weighted": np.nan, "qwk": np.nan, "mae": np.nan, "support": 0}
    
    # Clean: convert to numeric, filter valid range [1-5]
    df_clean = df.copy()
    df_clean["final_decision_num"] = pd.to_numeric(df_clean["final_decision"], errors="coerce")
    df_clean["model_level_num"] = pd.to_numeric(df_clean["model_level"], errors="coerce")
    
    df_clean = df_clean[
        (df_clean["final_decision_num"].between(1, 5)) &
        (df_clean["model_level_num"].between(1, 5))
    ]
    
    if df_clean.empty:
        return {"f1_macro": np.nan, "f1_weighted": np.nan, "qwk": np.nan, "mae": np.nan, "support": 0}
    
    y_true = df_clean["final_decision_num"].astype(int)
    y_pred = df_clean["model_level_num"].astype(int)
    
    return {
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        "mae": mean_absolute_error(y_true, y_pred),
        "support": len(df_clean),
    }

# Compute metrics per fold for each variant
metrics_rows = []
for variant in VARIANTS:
    variant_df = all_preds[all_preds["variant"] == variant]
    for fold_id in FOLDS:
        fold_df = variant_df[variant_df["fold"] == fold_id]
        metrics = compute_metrics(fold_df)
        metrics_rows.append({
            "variant": variant,
            "fold": fold_id,
            **metrics
        })

metrics_df = pd.DataFrame(metrics_rows)
print("✅ Computed metrics for all folds:")
display(metrics_df)

# Cell 4: Aggregate Metrics Across Folds

In [ ]:
# Cell 4: Aggregate Metrics Across Folds
# Mean and Std per variant
agg_mean = metrics_df.groupby("variant")[["f1_macro", "f1_weighted", "qwk", "mae"]].mean().round(4)
agg_std = metrics_df.groupby("variant")[["f1_macro", "f1_weighted", "qwk", "mae"]].std().round(4)

print("=" * 70)
print("AGGREGATE RESULTS ACROSS ALL FOLDS (K=5)")
print("=" * 70)
print("\n📊 Mean metrics:")
display(agg_mean)

print("\n📊 Standard deviations (across folds):")
display(agg_std)

# Combined "Mean ± Std" format
stability = agg_mean.copy()
for col in ["f1_macro", "f1_weighted", "qwk", "mae"]:
    stability[col] = agg_mean[col].astype(str) + " ± " + agg_std[col].astype(str)

print("\n📊 Stability (Mean ± Std):")
display(stability)

# Cell 5: Helper Functions for Hedges' g (Repeated Measures)


In [ ]:
# Cell 5: Helper Functions for Hedges' g (Repeated Measures)
def hedges_g_repeated_measures(method1_scores, method2_scores):
    """
    Calculate Hedges' g for repeated measures using Goulet-Pelletier & Cousineau (2018).
    
    Returns dict with S_p, cohens_d, nu, J, hedges_g, mean_diff
    """
    m1 = np.asarray(method1_scores, dtype=float)
    m2 = np.asarray(method2_scores, dtype=float)
    
    n = len(m1)
    
    # Step 1: Pooled Standard Deviation
    s1 = np.std(m1, ddof=1)
    s2 = np.std(m2, ddof=1)
    S_p = np.sqrt((s1**2 + s2**2) / 2)
    
    # Step 2: Cohen's d
    M1 = np.mean(m1)
    M2 = np.mean(m2)
    cohens_d = (M1 - M2) / S_p if S_p > 0 else 0
    
    # Step 3: Correction factor J (repeated measures)
    nu = 2 * (n - 1)
    J = 1 - (3 / (4 * nu - 1))
    
    # Step 4: Hedges' g
    hedges_g = cohens_d * J
    
    return {
        "S_p": S_p,
        "M1": M1,
        "M2": M2,
        "cohens_d": cohens_d,
        "nu": nu,
        "J": J,
        "hedges_g": hedges_g,
        "mean_diff": M1 - M2,
    }

def paired_ttest_with_hedges_g(a, b):
    """
    Paired t-test with Hedges' g effect size for repeated measures.
    
    Returns dict with t_stat, p_value, mean_diff, CI, hedges_g, effect_size_category, n_pairs
    """
    a, b = np.asarray(a, float), np.asarray(b, float)
    diffs = a - b
    n = len(diffs)
    
    if n < 2:
        return None
    
    mean_diff = np.mean(diffs)
    std_diff = np.std(diffs, ddof=1)
    se = std_diff / np.sqrt(n)
    
    # Paired t-test
    t_stat, p_val = stats.ttest_rel(a, b)
    
    # Calculate Hedges' g
    hedges_result = hedges_g_repeated_measures(a, b)
    
    # 95% CI
    ci = stats.t.interval(0.95, df=n-1, loc=mean_diff, scale=se)
    
    # Effect size categorization
    abs_g = abs(hedges_result["hedges_g"])
    if abs_g < 0.2:
        effect_cat = "negligible"
    elif abs_g < 0.5:
        effect_cat = "small"
    elif abs_g < 0.8:
        effect_cat = "medium"
    else:
        effect_cat = "large"
    
    return {
        "t_stat": t_stat,
        "p_value": p_val,
        "mean_diff": mean_diff,
        "ci_low": ci[0],
        "ci_high": ci[1],
        "S_p": hedges_result["S_p"],
        "cohens_d": hedges_result["cohens_d"],
        "hedges_g": hedges_result["hedges_g"],
        "effect_size_category": effect_cat,
        "n_pairs": n,
    }

# Cell 6: Paired T-Tests - Variant B vs Variant A


In [ ]:
# Cell 6: Paired T-Tests - Variant B vs Variant A
METRICS_TO_TEST = ["f1_macro", "f1_weighted", "qwk", "mae"]

# Extract metric pairs by fold
results_rows = []

for metric in METRICS_TO_TEST:
    # Get values for each variant across folds
    var_A_values = metrics_df[metrics_df["variant"] == "variant_A"][metric].values
    var_B_values = metrics_df[metrics_df["variant"] == "variant_B"][metric].values
    
    # Run paired t-test
    test_result = paired_ttest_with_hedges_g(var_B_values, var_A_values)
    
    # Append result
    results_rows.append({
        "metric": metric,
        **test_result,
        "significant": "✅ Yes" if test_result["p_value"] < 0.05 else "❌ No"
    })

ttest_results = pd.DataFrame(results_rows)

print("=" * 100)
print("PAIRED T-TESTS: Variant B (Skills-Aware) vs Variant A (Definitions Only)")
print("H0: No difference in metric values between variants")
print("=" * 100)

for idx, row in ttest_results.iterrows():
    metric = row["metric"]
    print(f"\n📊 {metric.upper()}")
    print(f"  t-statistic: {row['t_stat']:.4f}")
    print(f"  p-value: {row['p_value']:.4f}  {row['significant']}")
    print(f"  Mean difference (B - A): {row['mean_diff']:.4f}")
    print(f"  95% CI: [{row['ci_low']:.4f}, {row['ci_high']:.4f}]")
    print(f"  Pooled SD: {row['S_p']:.4f}")
    print(f"  Cohen's d: {row['cohens_d']:.4f}")
    print(f"  Hedges' g: {row['hedges_g']:.4f}  ({row['effect_size_category']})")
    print(f"  n pairs: {row['n_pairs']}")

# Cell 7: Results Summary Table

In [ ]:
# Cell 7: Results Summary Table
print("\n" + "=" * 100)
print("SUMMARY TABLE: PAIRED T-TEST RESULTS")
print("=" * 100)

summary_df = ttest_results[[
    "metric", "t_stat", "p_value", "mean_diff", 
    "hedges_g", "effect_size_category", "n_pairs", "significant"
]].copy()
summary_df["t_stat"] = summary_df["t_stat"].round(4)
summary_df["p_value"] = summary_df["p_value"].round(4)
summary_df["mean_diff"] = summary_df["mean_diff"].round(4)
summary_df["hedges_g"] = summary_df["hedges_g"].round(4)

display(summary_df)

# Save results
OUT_CSV = RAG_BASE / "results" / "rag_evaluation_ttest_results.csv"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
ttest_results.to_csv(OUT_CSV, index=False)
print(f"\n✅ Results saved to: {OUT_CSV}")